In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'run_fast_causal_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (str(PACKAGE_ROOT / 'src'), RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from calcium_transient_rising_flank.checkpointing import format_progress

# Environment setup: uv sync --frozen --all-extras
RUN_PCMCIPLUS = True
OUTPUT_DIR = PACKAGE_ROOT / 'outputs/causal_baselines/pcmciplus_motorneurons'
command = [RUNNER_PYTHON, 'examples/run_fast_causal_baselines.py', '--dataset', 'motorneurons', '--algorithm', 'pcmciplus', '--output-dir', str(OUTPUT_DIR), '--max-lag', '3', '--resume']
print(f'[notebook] launching: {shlex.join(command)}', flush=True)
if RUN_PCMCIPLUS:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
state_path = OUTPUT_DIR / 'motorneurons_pcmciplus_resume.json'
if state_path.exists():
    state = json.loads(state_path.read_text())
    print(format_progress(int(state.get('completed_units', 0)), max(1, int(state.get('total_units', 1))), label='PCMCI+ motorneuron fits'))
summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    print(summary_path.read_text())
